# 🔍 Wikipedia Query Search (with Infobox extraction)
Enter a query and get:
1. Clean top points from the intro summary
2. Key facts from the page's **infobox** table (e.g. Incumbent, Formation, Residence, Salary, etc.)

In [1]:
# Step 1: Install dependencies (bs4 is usually pre-installed in Colab, this just ensures it)
!pip install -q beautifulsoup4 lxml

In [2]:
# Step 2: Imports
import requests
import re
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "ColabWikiSearch/1.0 (educational use)"}

In [3]:
# Step 3: Cleaning / preprocessing helpers
def clean_text(text):
    text = re.sub(r"\[\d+\]", "", text)                          # remove [1], [2] citation marks
    text = re.sub(r"\[[a-zA-Z ]+\]", "", text)                   # remove [citation needed] etc.
    text = re.sub(r"\(/[^)]*\)", "", text)                        # remove IPA pronunciation
    text = re.sub(r"\([^)]*listen[^)]*\)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text)                              # collapse whitespace
    return text.strip()


def split_into_points(text, max_points=8):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    points = [s.strip() for s in sentences if len(s.strip()) > 25]
    return points[:max_points]

In [4]:
# Step 4: Search for the top matching page title
def get_top_title(query):
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": 1,
    }
    r = requests.get(url, params=params, headers=HEADERS, timeout=10)
    r.raise_for_status()
    results = r.json().get("query", {}).get("search", [])
    return results[0]["title"] if results else None

In [5]:
# Step 5: Get plain-text summary (intro paragraph) via REST API
def get_page_summary(title):
    safe_title = title.replace(" ", "_")
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{safe_title}"
    r = requests.get(url, headers=HEADERS, timeout=10)
    if r.status_code != 200:
        return None
    data = r.json()
    return {
        "title": data.get("title", title),
        "extract": data.get("extract", ""),
        "url": data.get("content_urls", {}).get("desktop", {}).get("page", f"https://en.wikipedia.org/wiki/{safe_title}"),
    }

In [31]:
# Step 6: Get the page's rendered HTML and extract the infobox table (now includes image)
def get_infobox(title):
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "parse",
        "page": title,
        "prop": "text",
        "format": "json",
        "redirects": 1,
    }
    r = requests.get(url, params=params, headers=HEADERS, timeout=10)
    if r.status_code != 200:
        return {}, None
    data = r.json()
    html = data.get("parse", {}).get("text", {}).get("*", "")
    if not html:
        return {}, None

    soup = BeautifulSoup(html, "lxml")
    infobox = soup.find("table", class_="infobox")
    if not infobox:
        return {}, None

    # --- Grab the main photo: Incumbent-style row -> infobox-image cell -> any image in table ---
    image_url = None

    # 1) Incumbent-style merged cell (e.g. Presidents/PMs with "Incumbent" bold label)
    for row in infobox.find_all("tr"):
        full_cell = row.find("td", class_="infobox-full-data")
        if full_cell and full_cell.find("div", style=lambda s: s and "font-weight:bold" in s):
            img_tag = full_cell.find("img")
            if img_tag and img_tag.get("src"):
                src = img_tag["src"]
                image_url = "https:" + src if src.startswith("//") else src
            break

    # 2) Fallback: standard infobox-image cell (e.g. biography infoboxes like Salman Khan)
    if not image_url:
        image_cell = infobox.find("td", class_="infobox-image")
        if image_cell:
            img_tag = image_cell.find("img")
            if img_tag and img_tag.get("src"):
                src = img_tag["src"]
                image_url = "https:" + src if src.startswith("//") else src

    # 3) Fallback: first image anywhere in the infobox (e.g. biota/taxobox tables, no special classes)
    if not image_url:
        img_tag = infobox.find("img")
        if img_tag and img_tag.get("src"):
            src = img_tag["src"]
            image_url = "https:" + src if src.startswith("//") else src

    facts = {}
    for row in infobox.find_all("tr"):
        for sup in row.find_all("sup"):
            sup.decompose()

        label_cell = row.find("th", class_="infobox-label")
        value_cell = row.find("td", class_="infobox-data")

        if label_cell and value_cell:
            label = clean_text(label_cell.get_text(" ", strip=True))
            value = clean_text(value_cell.get_text(" | ", strip=True))
            if label and value:
                facts[label] = value
            continue

        full_cell = row.find("td", class_="infobox-full-data")
        if full_cell:
            bold_div = full_cell.find("div", style=lambda s: s and "font-weight:bold" in s)
            if bold_div:
                bold_text = clean_text(bold_div.get_text(" ", strip=True))
                full_text = clean_text(full_cell.get_text(" ", strip=True))
                words = bold_text.split(" ", 1)
                if len(words) == 2:
                    label, name = words
                    remainder = full_text.replace(bold_text, "", 1).strip()
                    value = f"{name} ({remainder})" if remainder else name
                    facts[label] = value
            continue

        # --- Case C: plain td/td pairs, no class attrs (e.g. taxobox/biota tables) ---
        plain_cells = row.find_all("td", recursive=False)
        if len(plain_cells) == 2:
            label = clean_text(plain_cells[0].get_text(" ", strip=True)).rstrip(":")
            value = clean_text(plain_cells[1].get_text(" | ", strip=True))
            if label and value and len(label) < 30:  # avoid picking up stray long rows
                facts[label] = value

    return facts, image_url

In [7]:
# Step 7: Main function tying it all together
def wikipedia_search(query, num_points=6):
    print(f"\n🔎 Searching Wikipedia for: '{query}'\n")

    try:
        title = get_top_title(query)
        if not title:
            print("No results found.")
            return

        page = get_page_summary(title)
        if not page or not page["extract"]:
            print("Could not retrieve a summary for this page.")
            return

        print(f"📄 Title: {page['title']}")
        print(f"🔗 URL: {page['url']}\n")

        # --- Infobox facts + image ---
        facts, image_url = get_infobox(title)

        if image_url:
            from IPython.display import Image, display
            display(Image(url=image_url, width=200))

        if facts:
            print("📋 Key Facts (Infobox):\n")
            for label, value in facts.items():
                print(f"• {label}: {value}")
            print()

        # --- Summary points ---
        cleaned = clean_text(page["extract"])
        points = split_into_points(cleaned, max_points=num_points)

        print("📌 Top Points:\n")
        if points:
            for i, point in enumerate(points, 1):
                print(f"{i}. {point}")
        else:
            print(cleaned)

    except requests.exceptions.RequestException as e:
        print(f"Network error: {e}")
    except Exception as e:
        print(f"Error: {e}")

In [42]:
# Step 8: Run it — enter your query below
query = input("Enter your search query: ")
wikipedia_search(query, num_points=6)

Enter your search query: smallest country by size

🔎 Searching Wikipedia for: 'smallest country by size'

📄 Title: List of countries and dependencies by area
🔗 URL: https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_area

📌 Top Points:

1. This is a list of the world's countries and their dependencies, ranked by total area, including land and water.
